In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df = spark.read \
    .parquet('fhvhv_tripdata_2021-01.parquet')

In [ ]:
df.schema

In [ ]:
df.show(5)

In [ ]:
df = df.repartition(24)

In [ ]:
df.write \
    .mode('overwrite') \
    .parquet('partitioned/')

# 2. Read the partitioned data (Next day)

In [ ]:
df = spark.read \
    .parquet('partitioned/')

In [ ]:
df.show(5)

In [ ]:
df.printSchema()

In [ ]:
df.select('hvfhs_license_num').filter(df['hvfhs_license_num'] == 'HV0003').show()

In [ ]:
from pyspark.sql import functions as F

In [ ]:
def crazy_stuff(dispatching_base_num: str):
    """
    This function does some crazy stuff
    :param dispatching_base_num: 
    :return: 
    """
    num = int(dispatching_base_num[1:])
    if num % 2 == 0:
        return 'even'
    else:
        return 'odd'

crazy_stuff_udf = F.udf(crazy_stuff, returnType='string')

In [ ]:
df \
    .withColumn('pickup_date', F.to_date('pickup_datetime')) \
    .withColumn('pickup_hour', F.hour('pickup_datetime')) \
    .withColumn('crazy_base_num', crazy_stuff_udf('dispatching_base_num')) \
    .select('pickup_date', 'pickup_hour', 'crazy_base_num') \
    .show(5)